# Predictive Maintenance Signal Forecasting

## Project Overview

Forecasts **daily machine vibration levels** (mm/s) over a 14-day horizon. Synthetic data spans 2 years with weekly utilization patterns, gradual degradation, and periodic maintenance resets.

**Models**: Naive, LazyPredict, FLAML, StatsForecast. **Horizon**: 14 days.

## Learning Objectives

1. Understand time-series patterns (trend, seasonality, noise)
2. Build naive and seasonal-naive baselines
3. Engineer lag and rolling features for tabular ML
4. Use LazyPredict for quick regression benchmarking
5. Apply FLAML AutoML (excluding XGBoost due to compatibility)
6. Use StatsForecast (AutoARIMA, AutoETS, SeasonalNaive)
7. Compare all approaches with MAE / RMSE / MAPE

## Problem Statement

Given historical daily vibration readings, predict the next 14 days. Maintenance teams use vibration forecasts to detect degradation trends and schedule maintenance before catastrophic failure.

## Why This Project Matters

Unplanned downtime costs manufacturing $50B+ annually. Vibration analysis is the gold standard for rotating equipment health monitoring. Forecasting vibration trends enables condition-based maintenance — replacing parts just before failure, not on fixed schedules.

## Dataset Overview

Synthetic dataset:
- 730 days (2 years) of daily average vibration
- Weekly utilization pattern (higher on production days)
- Gradual degradation trend (bearing/gear wear)
- Periodic maintenance resets (vibration drops after service)
- Random shock events

| Column | Description |
|--------|-------------|
| `date` | Date |
| `vibration_mm_s` | Daily RMS vibration velocity (mm/s) |

## Dataset Source and License Notes

Synthetically generated for educational purposes. No external download required.

## Environment Setup

In [1]:
import subprocess, importlib
for pkg in ['numpy', 'pandas', 'matplotlib', 'scikit-learn', 'statsforecast', 'flaml', 'lazypredict']:
    try:
        importlib.import_module(pkg.replace('-', '_').split('[')[0])
    except ImportError:
        subprocess.check_call(['pip', 'install', '-q', pkg])
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## Imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import mean_absolute_error, mean_squared_error
print("Imports complete.")

Imports complete.

## Configuration / Constants

In [3]:
SEED = 42
N_POINTS = 730
HORIZON = 14
VAL_SIZE = 14
TEST_SIZE = 14
SEASON_LENGTH = 7
FREQ = 'D'
TARGET = 'vibration_mm_s'
np.random.seed(SEED)
print(f"Config: {N_POINTS} points, horizon={HORIZON}, season={SEASON_LENGTH}")

Config: 730 points, horizon=14, season=7


## Dataset Generation

In [4]:
dates = pd.date_range(start='2022-01-01', periods=N_POINTS, freq='D')
t = np.arange(N_POINTS)

# Sawtooth degradation with maintenance resets every ~120 days
base = np.zeros(N_POINTS)
maintenance_interval = 120
for i in range(N_POINTS):
    cycle_pos = i % maintenance_interval
    base[i] = 2.5 + 0.015 * cycle_pos  # gradual increase within cycle

weekly = np.zeros(N_POINTS)
for i in range(N_POINTS):
    dow = dates[i].dayofweek
    if dow <= 4: weekly[i] = 0.3  # production load
    elif dow == 5: weekly[i] = -0.2
    else: weekly[i] = -0.5  # idle

# Shock events
shocks = np.where(np.random.random(N_POINTS) < 0.05, np.random.uniform(0.5, 1.5, N_POINTS), 0)
noise = np.random.normal(0, 0.15, N_POINTS)

vibration_mm_s = base + weekly + shocks + noise
vibration_mm_s = np.maximum(vibration_mm_s, 0.5).round(2)

df = pd.DataFrame({'date': dates, 'vibration_mm_s': vibration_mm_s})
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (730, 2)


,date,vibration_mm_s
0,2022-01-01,2.55
1,2022-01-02,1.84
2,2022-01-03,2.88
3,2022-01-04,2.73
4,2022-01-05,2.85


## Data Validation Checks

In [5]:
assert df.isnull().sum().sum() == 0, "Missing values"
assert (df[TARGET] > 0).all(), "Non-positive values"
assert df['date'].is_monotonic_increasing, "Not sorted"
assert len(df) == N_POINTS, "Row count mismatch"
print("All validation checks passed.")

All validation checks passed.


## Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].plot(df['date'], df[TARGET], linewidth=0.6)
axes[0, 0].set_title('vibration_mm_s Over Time')
axes[0, 1].hist(df[TARGET], bins=30, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Distribution')
from pandas.plotting import autocorrelation_plot
autocorrelation_plot(df[TARGET], ax=axes[1, 0])
axes[1, 0].set_xlim(0, min(60, N_POINTS // 2))
axes[1, 0].set_title('Autocorrelation')
rw = min(SEASON_LENGTH * 2, N_POINTS // 4)
axes[1, 1].plot(df['date'], df[TARGET].rolling(rw).mean())
axes[1, 1].set_title(f'Rolling {rw}-period Mean')
plt.tight_layout()
plt.savefig('eda.png', dpi=100, bbox_inches='tight')
plt.show()
print("EDA complete.")

EDA complete.


## Target Analysis

In [7]:
print("vibration_mm_s Statistics:")
print(df[TARGET].describe())
print(f"\nCV: {df[TARGET].std() / df[TARGET].mean():.3f}")
print(f"Skewness: {df[TARGET].skew():.3f}")

vibration_mm_s Statistics:
count    730.000000
mean       3.553822
std        0.663800
min        1.840000
25%        3.080000
50%        3.550000
75%        4.020000
max        5.760000
Name: vibration_mm_s, dtype: float64

CV: 0.187
Skewness: 0.133


## Train / Validation / Test Split Strategy

Time-aware split (no shuffling):
- **Train**: all except last 28
- **Validation**: 14 points
- **Test**: last 14 points

In [8]:
train = df.iloc[:-(VAL_SIZE + TEST_SIZE)].copy()
val = df.iloc[-(VAL_SIZE + TEST_SIZE):-TEST_SIZE].copy()
test = df.iloc[-TEST_SIZE:].copy()
print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

Train: 702, Val: 14, Test: 14


## Preprocessing Strategy

Minimal preprocessing — tree models handle raw features. Features created next.

In [9]:
df_full = df.copy().sort_values('date').reset_index(drop=True)
print('Data ready.')

Data ready.


## Feature Engineering

In [10]:
def create_features(data):
    d = data.copy()
    d['dow'] = d['date'].dt.dayofweek
    d['month'] = d['date'].dt.month
    d['day'] = d['date'].dt.day
    d['week_of_year'] = d['date'].dt.isocalendar().week.astype(int)
    for lag in [1, 7, 14, 28]:
        d[f'lag_{lag}'] = d[TARGET].shift(lag)
    for w in [7, 14, 28]:
        d[f'rmean_{w}'] = d[TARGET].shift(1).rolling(w).mean()
        d[f'rstd_{w}'] = d[TARGET].shift(1).rolling(w).std()
    return d

df_feat = create_features(df_full).dropna().reset_index(drop=True)
feature_cols = [c for c in df_feat.columns if c not in ['date', TARGET]]
print(f"Features: {len(feature_cols)} columns, {len(df_feat)} rows")

Features: 14 columns, 702 rows


## Baseline Model — Naive Forecasts

In [11]:
def calc_metrics(y_true, y_pred, name=""):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1))) * 100
    print(f"{name:30s} | MAE: {mae:10.1f} | RMSE: {rmse:10.1f} | MAPE: {mape:5.2f}%")
    return {'model': name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

results = []
naive_pred = np.full(TEST_SIZE, train[TARGET].iloc[-1])
results.append(calc_metrics(test[TARGET].values, naive_pred, "Naive (Last Value)"))

src = df_full[TARGET].values
ti = len(df_full) - TEST_SIZE
sn_pred = src[ti - SEASON_LENGTH:ti - SEASON_LENGTH + TEST_SIZE]
results.append(calc_metrics(test[TARGET].values, sn_pred, f"Seasonal Naive ({SEASON_LENGTH})"))

Naive (Last Value)             | MAE:        0.9 | RMSE:        1.0 | MAPE: 33.18%
Seasonal Naive (7)             | MAE:        0.9 | RMSE:        1.2 | MAPE: 33.16%


## LazyPredict Benchmark (Lag-Feature Tabular)

In [12]:
from lazypredict.Supervised import LazyRegressor

ct = len(df_feat) - TEST_SIZE
cv = ct - VAL_SIZE
X_tr = df_feat.iloc[:cv][feature_cols]
y_tr = df_feat.iloc[:cv][TARGET]
X_va = df_feat.iloc[cv:ct][feature_cols]
y_va = df_feat.iloc[cv:ct][TARGET]

reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None, predictions=True)
lp_models, lp_preds = reg.fit(X_tr, X_va, y_tr, y_va)
print("\nLazyPredict Top 10:")
print(lp_models.head(10).to_string())


LazyPredict Top 10:
                            Adjusted R-Squared   R-Squared      RMSE  Time Taken
Model                                                                           
KernelRidge                        2049.301156 -156.561627  3.531727    0.047748
GaussianProcessRegressor           1048.827100  -79.602085  2.526010    0.041121
ExtraTreeRegressor                  222.147126  -16.011317  1.160462    0.007645
MLPRegressor                        128.483804   -8.806446  0.881085    0.271003
QuantileRegressor                    92.832818   -6.064063  0.747806    0.061834
ElasticNet                           89.133026   -5.779464  0.732588    0.014205
Lasso                                89.133026   -5.779464  0.732588    0.005903
LassoLars                            89.133026   -5.779464  0.732588    0.006479
DummyRegressor                       89.133026   -5.779464  0.732588    0.005691
PassiveAggressiveRegressor           34.122540   -1.547888  0.449109    0.007205


## FLAML AutoML Run

In [13]:
from flaml import AutoML

automl = AutoML()
automl.fit(
    X_train=X_tr, y_train=y_tr,
    task='regression', time_budget=30, metric='mae',
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
print(f"FLAML best: {automl.best_estimator}")
flaml_val = automl.predict(X_va)
results.append(calc_metrics(y_va.values, flaml_val, f"FLAML ({automl.best_estimator})"))

X_te = df_feat.iloc[ct:][feature_cols]
y_te = df_feat.iloc[ct:][TARGET]
flaml_test = automl.predict(X_te)
results.append(calc_metrics(y_te.values, flaml_test, f"FLAML Test ({automl.best_estimator})"))

FLAML best: catboost
FLAML (catboost)               | MAE:        0.1 | RMSE:        0.1 | MAPE:  1.94%
FLAML Test (catboost)          | MAE:        0.4 | RMSE:        0.6 | MAPE: 16.84%


## StatsForecast — Statistical Forecasting

In [14]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, SeasonalNaive as SFSeasonalNaive

sf_df = df_full[['date', TARGET]].copy()
sf_df.columns = ['ds', 'y']
sf_df['unique_id'] = 'series1'
sf_df = sf_df[['unique_id', 'ds', 'y']]

sf_train = sf_df.iloc[:-TEST_SIZE]
sf = StatsForecast(
    models=[AutoARIMA(season_length=SEASON_LENGTH), AutoETS(season_length=SEASON_LENGTH),
            SFSeasonalNaive(season_length=SEASON_LENGTH)],
    freq=FREQ, n_jobs=1
)
sf.fit(sf_train)
sf_preds = sf.predict(h=TEST_SIZE).reset_index()

y_test_sf = test[TARGET].values
for col in ['AutoARIMA', 'AutoETS', 'SeasonalNaive']:
    if col in sf_preds.columns:
        results.append(calc_metrics(y_test_sf, sf_preds[col].values, f"SF {col}"))
print("StatsForecast complete.")

SF AutoARIMA                   | MAE:        1.1 | RMSE:        1.2 | MAPE: 40.23%
SF AutoETS                     | MAE:        1.2 | RMSE:        1.3 | MAPE: 44.71%
SF SeasonalNaive               | MAE:        1.2 | RMSE:        1.3 | MAPE: 44.53%
StatsForecast complete.


## Top 3 Model Selection

In [15]:
results_df = pd.DataFrame(results).sort_values('MAE').reset_index(drop=True)
print("All Results:")
print(results_df.to_string(index=False))
print("\nTop 3:")
print(results_df.head(3).to_string(index=False))

All Results:
                model      MAE     RMSE      MAPE
     FLAML (catboost) 0.082508 0.103177  1.940543
FLAML Test (catboost) 0.429429 0.636711 16.835546
   Seasonal Naive (7) 0.882857 1.170220 33.157592
   Naive (Last Value) 0.930000 0.980153 33.175166
         SF AutoARIMA 1.054131 1.206169 40.228924
     SF SeasonalNaive 1.157143 1.341800 44.527215
           SF AutoETS 1.164906 1.343212 44.711942

Top 3:
                model      MAE     RMSE      MAPE
     FLAML (catboost) 0.082508 0.103177  1.940543
FLAML Test (catboost) 0.429429 0.636711 16.835546
   Seasonal Naive (7) 0.882857 1.170220 33.157592


## Final Training and Evaluation of Top 3

In [16]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test['date'].values, test[TARGET].values, 'ko-', label='Actual', ms=4)
ax.plot(test['date'].values, flaml_test, 's--', label=f'FLAML ({automl.best_estimator})', ms=4)
for col in ['AutoARIMA', 'AutoETS']:
    if col in sf_preds.columns:
        ax.plot(test['date'].values[:len(sf_preds)], sf_preds[col].values, 'o--', label=f'SF {col}', ms=4)
ax.set_title('Forecast vs Actual — Test Set')
ax.legend()
plt.tight_layout()
plt.savefig('forecast_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## Error Analysis

In [17]:
errors = test[TARGET].values - flaml_test
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(len(errors)), errors, alpha=0.7)
axes[0].set_title('Residuals (Actual - FLAML)')
axes[0].axhline(y=0, color='r', linestyle='--')
axes[1].hist(errors, bins=10, edgecolor='black', alpha=0.7)
axes[1].set_title('Residual Distribution')
plt.tight_layout()
plt.savefig('error_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Mean residual: {np.mean(errors):.2f}, Std: {np.std(errors):.2f}")

Mean residual: -0.38, Std: 0.51


## Interpretation and Business Insight

- Vibration follows a sawtooth pattern: gradual increase → maintenance reset
- Production days (weekdays) show higher vibration due to load
- Shock events appear as sudden spikes
- The degradation rate between maintenance cycles is fairly constant
- Forecasting the trend enables optimal maintenance timing

## Limitations

1. Synthetic — real vibration data has much higher frequency (kHz)
2. Daily averages mask critical transient events
3. Single sensor — real systems use multiple vibration axes
4. No frequency spectrum analysis (bearing fault signatures)
5. No process variable correlation (load, speed, temperature)

## How to Improve This Project

1. Use high-frequency vibration data (per-second)
2. Add frequency domain features (FFT, envelope analysis)
3. Include process variables (RPM, load, temperature)
4. Label fault types for supervised classification
5. Use anomalib or PyOD for anomaly scoring

## Production Considerations

- Real-time vibration monitoring with trend alerting
- Integration with CMMS for work order generation
- Remaining useful life (RUL) estimation
- Multi-machine fleet health dashboards

## Common Mistakes

1. Using daily averages when transient events matter
2. Setting fixed vibration thresholds instead of trend-based alerts
3. Ignoring load and speed effects on vibration
4. Not separating normal wear from fault-induced vibration

## Mini Challenge / Exercises

1. Estimate the degradation rate (mm/s per day) between maintenance cycles
2. Predict when vibration will exceed an alarm threshold (e.g., 4.5 mm/s)
3. Add synthetic RPM data and correlate with vibration
4. Build a simple anomaly detector for shock events

## Final Summary / Key Takeaways

1. Vibration forecasting enables condition-based maintenance
2. Sawtooth degradation pattern is highly predictable
3. Maintenance timing should be based on trend, not fixed schedules
4. Real deployment needs high-frequency data and frequency analysis
5. Forecast-based anomaly detection outperforms fixed thresholds

In [18]:
import json
metrics = {
    'project': 'Predictive Maintenance Signal Forecasting',
    'best_model': results_df.iloc[0]['model'],
    'best_MAE': float(results_df.iloc[0]['MAE']),
    'best_RMSE': float(results_df.iloc[0]['RMSE']),
    'best_MAPE': float(results_df.iloc[0]['MAPE']),
    'all_results': results_df.to_dict('records')
}
with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved.")
print("\n=== Predictive Maintenance Signal Forecasting — Complete ===")

Metrics saved.

=== Predictive Maintenance Signal Forecasting — Complete ===
